In [3]:

import pandas as pd

# Part 1. Repository Mining

In [4]:
# Load the columns of interest from the pull request dataset
selected_columns = [
    "id", "number",  "title", "body","agent", "user", "state","created_at", "closed_at", "merged_at", "repo_url"]

pull_request_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v4/pull_request.parquet", columns=selected_columns)

pr_reviews_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v4/pr_reviews.parquet")
pr_comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v4/pr_comments.parquet")
pr_commits_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v4/pr_commits.parquet")

print(f"Number of PRs: {len(pull_request_df)}")

# Print the columns to verify the join
print(pr_reviews_df.columns.tolist())
print(pr_comments_df.columns.tolist())
print(pr_commits_df.columns.tolist())

Number of PRs: 71677
['id', 'pr_id', 'user', 'user_type', 'state', 'submitted_at', 'body']
['id', 'pr_id', 'user', 'user_id', 'user_type', 'created_at', 'body']
['sha', 'pr_id', 'author', 'committer', 'message']


In [5]:
# Joint the datasets based on the pull request ID
nb_reviews = pr_reviews_df.groupby("pr_id").size().rename("n_reviews")
nb_comments = pr_comments_df.groupby("pr_id").size().rename("n_comments")
nb_commits = pr_commits_df.groupby("pr_id").size().rename("n_commits")

pull_request_df = pull_request_df.merge(nb_comments, left_on="id", right_index=True, how="left")
pull_request_df = pull_request_df.merge(nb_reviews, left_on="id", right_index=True, how="left")
pull_request_df = pull_request_df.merge(nb_commits, left_on="id", right_index=True, how="left")


columns_to_fill= ["n_comments", "n_reviews", "n_commits"]
pull_request_df[columns_to_fill] = pull_request_df[columns_to_fill].fillna(0).astype(int)


# Part 2. Data Cleaning & Classification

In [6]:
# Remove duplicates
pr_counter = len(pull_request_df)
pull_request_df = pull_request_df.drop_duplicates(subset=['id'])
print(f"Number of PRs after removing duplicates: {len(pull_request_df)}")

Number of PRs after removing duplicates: 71677


In [7]:
#  Handle missing values
pull_request_df["is_merged"] = pull_request_df["merged_at"].notna()
pull_request_df["is_closed"] = pull_request_df["closed_at"].notna()

In [8]:
# Identify AI-related vs human
pull_request_df["contributor_type"] = pull_request_df["agent"].apply(
    lambda agent: "AI" if pd.notna(agent) else "Human"
)

In [9]:
# Print the final dataset 
pull_request_df.head()

,id,number,title,body,agent,user,state,created_at,closed_at,merged_at,repo_url,n_comments,n_reviews,n_commits,is_merged,is_closed,contributor_type
0,3369524944,9,feat: Add extensive list of EV charging tools ...,This pull request resolves the issue by adding...,Google_Jules,google-labs-jules[bot],closed,2025-08-30T21:33:07Z,2025-08-30T21:40:30Z,2025-08-30T21:40:30Z,https://api.github.com/repos/juherr/awesome-ev...,0,1,1,True,True,AI
1,3369556773,10,Add last activity date to GitHub links,This change adds the date of the last activity...,Google_Jules,google-labs-jules[bot],open,2025-08-30T22:20:29Z,NaN,NaN,https://api.github.com/repos/juherr/awesome-ev...,0,0,1,False,False,AI
2,3205908454,69,Pull request for issue #68,Fixes #68,Google_Jules,google-labs-jules[bot],closed,2025-07-06T02:00:56Z,2025-07-06T02:37:38Z,2025-07-06T02:37:38Z,https://api.github.com/repos/steve02081504/fount,0,0,2,True,True,AI
3,3449941258,2333,Use permanent cache for library manga covers,This change ensures that the `DisplayManga` fo...,Google_Jules,google-labs-jules[bot],closed,2025-09-24T15:31:19Z,2025-10-13T00:07:33Z,NaN,https://api.github.com/repos/nekomangaorg/Neko,1,0,2,False,True,AI
4,3450319162,2334,Optimize LibraryComposePresenter Performance,This change optimizes the performance of the l...,Google_Jules,google-labs-jules[bot],closed,2025-09-24T17:34:48Z,2025-09-25T11:50:54Z,2025-09-25T11:50:54Z,https://api.github.com/repos/nekomangaorg/Neko,1,1,6,True,True,AI
